# 01 · Preprocessing

Inspect → harmonize → clean → optionally reserve an unseen attack → split → fit on train → transform/write each split in batches → release that split.

One dataset per run. Initial loading, splitting, and fitting still need RAM. Saved Parquet files contain ready-to-use transformed features.

## 0. Setup and Imports

In [ ]:
PLATFORM = 'kaggle'  # 'colab' or 'kaggle'
assert PLATFORM in ('colab', 'kaggle')

In [ ]:
if PLATFORM == 'colab':
    %load_ext cudf.pandas
    %load_ext cuml.accel

    from google.colab import drive
    drive.mount('/content/drive')
    INPATH = '/content/drive/MyDrive/Masters/Datasets/NF-UQ-NIDS(Splitted_Dataset)'
    OUTPATH = '/content/drive/MyDrive/Masters/IDS_repo/My repo'

else : 
    INPATH = '/kaggle/input/datasets/omarahmed001/ids-datasets'
    OUTPATH = '/kaggle/working/project'
    

In [ ]:
%pip install -q "scikit-learn>=1.6"
from pathlib import Path
from fastai.tabular.all import df_shrink
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.frozen import FrozenEstimator
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.decomposition import PCA


In [ ]:
DATA_DIR = Path(INPATH)
FILES = {
    'NF_UNSW_NB15': 'NF-UNSW-NB15-v2.parquet',
    'NF_BOT_IOT': 'NF-BoT-IoT-v2.parquet',
    'NF_CSE_CIC_IDS2018': 'NF-CSE-CIC-IDS2018-v2.parquet',
    'NF_TON_IOT': 'NF-ToN-IoT-v2.parquet',
    'CIC-BoT-IoT': 'CIC-BoT-IoT-V2.parquet',
    'CSE-CIC-IDS2018': 'CSE-CIC-IDS2018.parquet',
    'UNSW-NB15': 'UNSW-NB15.parquet'
}

In [ ]:
DATASET = 'CIC-BoT-IoT'  # Change this to run one of the other datasets.
HOLD_OUT_ATTACK = False  # False: include every attack class in the 70/15/15 split.
HELD_OUT_ATTACK = 'Worms'  # Used only when HOLD_OUT_ATTACK is True.
SEED = 42

## 1. Load and inspect

In [ ]:
df = pd.read_parquet(DATA_DIR / FILES[DATASET])
print(DATASET, df.shape)
display(df.head())
display(df['Attack'].value_counts())

In [ ]:
df.Label.value_counts()

In [ ]:
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame({
        'Dtype': df.dtypes,
        'Unique Values': df.nunique()
    }))

- Multiple constant columns that will be dropped later

In [ ]:
df['Down/Up Ratio'].value_counts()

### Actual sample versus the full original dataset

Keep up to `MAX_PER_CLASS = 300_000` randomly selected rows per `Attack`; retain every row from classes smaller than the cap. The cells compare this actual sample with the full original data before encoding, cleaning, splitting, or fitted transformations. No second reference sample or 50,000-row diagnostic cap is used.

The class table measures the deliberate change in class proportions. Numerical features are compared within each Attack using KS distance: 0 means matching empirical CDFs; 0.02 means a maximum CDF gap of 2 percentage points. Nominal fields use total variation distance between category frequencies (0 means matching frequencies). Missing and infinite rates are reported separately. Both distances range from 0 to 1; no universal pass/fail cutoff is imposed. Read the complete tables in `numeric_distribution_report` and `categorical_distribution_report`.

These are descriptive comparisons. KS p-values are not reported because the sample is a subset of the original, many features are discrete, and a small p-value would not establish a practically important difference. Marginal agreement does not prove matching correlations, rare subgroups, or tails. Keeping every minority row gives matching raw distributions for that class; it does not add new independent observations.

`CHECK_SAMPLE_DISTRIBUTION = False` skips the feature comparisons while retaining the class table and releasing the original. Full-data comparisons may take time; they process one column at a time and print each class. Rerun **Load and inspect** before repeating sampling.

Sampling remains before the split in this workflow: validation/test therefore reflect the capped class mix. For evaluation at the original class proportions, split the original data first and cap training only. Preprocessing is still fitted after splitting.

In [ ]:
import gc

MAX_PER_CLASS = 300_000
CHECK_SAMPLE_DISTRIBUTION = True

# Retain the full original data until the comparison below is finished.
# This assignment does not copy the original DataFrame.
original_df = df
original_counts = original_df['Attack'].value_counts(dropna=False)

sampled_train = pd.concat(
    [
        group.sample(n=min(len(group), MAX_PER_CLASS), random_state=SEED)
        for _, group in original_df.groupby(
            'Attack', observed=True, dropna=False
        )
    ],
    ignore_index=True,
)
df = sampled_train.sample(frac=1, random_state=SEED).reset_index(drop=True)
del sampled_train

sample_counts = df['Attack'].value_counts(dropna=False)
class_comparison = pd.DataFrame({
    'Original rows': original_counts,
    'Sample rows': sample_counts,
}).fillna(0).astype('int64').rename_axis('Attack')
class_comparison['Original (%)'] = (
    100 * class_comparison['Original rows'] / len(original_df)
)
class_comparison['Sample (%)'] = (
    100 * class_comparison['Sample rows'] / len(df)
)
class_comparison['Difference (percentage points)'] = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
)
class_comparison['Rows retained (%)'] = (
    100 * class_comparison['Sample rows'] / class_comparison['Original rows']
)
print(f'Original: {len(original_df):,} rows | Actual sample: {len(df):,} rows')
display(class_comparison.round(4))
class_tv_distance = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
).abs().sum() / 200
print(f'Class-proportion TV distance: {class_tv_distance:.6f}')

In [ ]:
from scipy.stats import ks_2samp

# Nominal fields are compared by category frequencies, not numeric ordering.
# Add another dataset-specific nominal column here if needed.
sampling_nominal_columns = ['Protocol']

if CHECK_SAMPLE_DISTRIBUTION:
    categorical_features = [
        c for c in original_df.columns
        if c not in ('Label', 'Attack')
        and (c in sampling_nominal_columns
             or not pd.api.types.is_numeric_dtype(original_df[c]))
    ]
    numeric_features = [
        c for c in original_df.select_dtypes(include='number').columns
        if c not in ('Label', 'Attack') and c not in categorical_features
    ]

    numeric_results = []
    categorical_results = []

    for attack in original_counts.index:
        print(f'Comparing full original vs actual sample: {attack}', flush=True)
        original_mask = (original_df['Attack'].isna() if pd.isna(attack)
                         else original_df['Attack'].eq(attack))
        sample_mask = (df['Attack'].isna() if pd.isna(attack)
                       else df['Attack'].eq(attack))

        # Use every row in both datasets; process one feature at a time.
        for col in numeric_features:
            a = original_df.loc[original_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            b = df.loc[sample_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            finite_a = a[np.isfinite(a)]
            finite_b = b[np.isfinite(b)]

            distance = (
                ks_2samp(finite_a, finite_b, method='asymp').statistic
                if len(finite_a) and len(finite_b) else np.nan
            )
            numeric_results.append({
                'Attack': attack,
                'feature': col,
                'Original rows': len(a),
                'Sample rows': len(b),
                'KS_D': distance,
                'Original missing (%)': 100 * np.isnan(a).mean(),
                'Sample missing (%)': 100 * np.isnan(b).mean(),
                'Original infinite (%)': 100 * np.isinf(a).mean(),
                'Sample infinite (%)': 100 * np.isinf(b).mean(),
            })
            del a, b, finite_a, finite_b

        for col in categorical_features:
            original_frequencies = original_df.loc[
                original_mask, col
            ].value_counts(normalize=True, dropna=False)
            sample_frequencies = df.loc[
                sample_mask, col
            ].value_counts(normalize=True, dropna=False)
            frequencies = pd.concat([
                original_frequencies.rename('original'),
                sample_frequencies.rename('sample'),
            ], axis=1).fillna(0)
            tv_distance = (
                frequencies['original'] - frequencies['sample']
            ).abs().sum() / 2
            categorical_results.append({
                'Attack': attack,
                'feature': col,
                'TV_distance': tv_distance,
            })
            del original_frequencies, sample_frequencies, frequencies

        del original_mask, sample_mask

    numeric_distribution_report = pd.DataFrame(numeric_results, columns=[
        'Attack', 'feature', 'Original rows', 'Sample rows', 'KS_D',
        'Original missing (%)', 'Sample missing (%)',
        'Original infinite (%)', 'Sample infinite (%)',
    ]).sort_values('KS_D', ascending=False)
    categorical_distribution_report = pd.DataFrame(
        categorical_results, columns=['Attack', 'feature', 'TV_distance']
    ).sort_values('TV_distance', ascending=False)

    print('Largest numerical distribution differences (KS distance):')
    display(numeric_distribution_report.head(30))
    if not numeric_distribution_report.empty:
        display(numeric_distribution_report.groupby('Attack', dropna=False)[
            'KS_D'
        ].agg(['median', 'max']))
    print('Categorical distribution differences (total variation distance):')
    display(categorical_distribution_report)
else:
    print('Within-class distribution comparisons disabled.')

# Release the full original before encoding, splitting, and preprocessing.
# The sampled df and the small comparison reports remain available.
del original_df
gc.collect()
print('Sampling comparison finished; original data released from RAM.')

In [ ]:
# 1. Summarize the feature distances separately for every class.
# Multiply by 100 to express CDF gaps in percentage points.
numeric_summary = (
    numeric_distribution_report
    .groupby('Attack', observed=True, dropna=False)['KS_D']
    .agg(['median', 'max'])
    .mul(100)
    .rename(columns={
        'median': 'Median CDF gap across features (pp)',
        'max': 'Largest CDF gap (pp)',
    })
)

# Start with ALL classes, then attach their numerical results.
sample_distribution_summary = class_comparison[
    ['Original rows', 'Sample rows']
].join(numeric_summary)

print('SUMMARY — one row per class')
print('pp = percentage points; smaller gaps mean closer distributions.')

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
):
    display(sample_distribution_summary.round(4))


# 2. Show five features for EACH class.
# This limits features per class, not the number of classes displayed.
for attack in sample_distribution_summary.index:
    mask = (
        numeric_distribution_report['Attack'].isna()
        if pd.isna(attack)
        else numeric_distribution_report['Attack'].eq(attack)
    )

    details = (
        numeric_distribution_report.loc[mask, ['feature', 'KS_D']]
        .sort_values('KS_D', ascending=False)
        .head(5)
        .copy()
    )
    details['Max CDF gap (pp)'] = 100 * details['KS_D']

    print(f'\n{attack} — five largest numerical feature differences:')
    display(details.reset_index(drop=True).round(6))


# 3. Show categorical results for ALL classes.
print('\nCATEGORY FREQUENCIES — zero means matching proportions.')

with pd.option_context('display.max_rows', None):
    display(
        categorical_distribution_report
        .sort_values(['Attack', 'feature'])
        .reset_index(drop=True)
        .round(6)
    )

## 2. Encoding

| Feature | Unique values | Type | Handling | 
|---|---:|---|---:|
| `Protocol` | 3 | Nominal | One Hot Encoding

In [ ]:
col_to_one_hot = [
    'Protocol'
]

IMPUTER_FIT_ROWS = 500_000
PCA_FIT_ROWS = IMPUTER_FIT_ROWS  # None: full train; integer: sample cap.
PCA_COMPONENTS = None  # None disables PCA; e.g. 10 reduces scaled measurements to 10 columns.
NUMERIC_SCALER = QuantileTransformer(output_distribution='normal', random_state=SEED)
PREPROCESSING_VERBOSE = True
PRINT_EVERY_BATCHES = 10  # Export reporting only.


In [ ]:
col_categorical = col_to_one_hot 

## 3. Dataset Shrinking

In [ ]:
import warnings
from fastai.tabular.core import df_shrink_dtypes

results = []

for col in df.select_dtypes(include="number").columns:
    # Inspect one column without copying or modifying the DataFrame.
    skip = [c for c in df.columns if c != col]

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)

        proposed = df_shrink_dtypes(
            df,
            skip=skip,
            obj2cat=False,
            int2uint=False,
        )

    if caught:
        results.append({
            "column": col,
            "original_dtype": str(df[col].dtype),
            "proposed_dtype": str(proposed.get(col, df[col].dtype)),
            "minimum": df[col].min(),
            "maximum": df[col].max(),
            "warning": "; ".join(str(w.message) for w in caught),
        })

display(pd.DataFrame(results))

In [ ]:
df.columns = df.columns.str.strip()
df['Attack'] = df['Attack'].astype("string[pyarrow]").str.strip()

In [ ]:
df = df_shrink(df, obj2cat=False, int2uint=False)
with pd.option_context('display.max_rows', None):
    display(df.dtypes)

## 4. Clean

In [ ]:
# Identify constant columns
constant_cols = df.columns[df.nunique(dropna=False) == 1]

print("Constant columns:", constant_cols.tolist())
print("Number of constant columns:", len(constant_cols))

# Drop constant columns
#df.drop(columns=constant_cols, inplace=True)

In [ ]:
# Drop constant and unnecessary columns
#df.drop(constant_cols)

In [ ]:
print(df.duplicated().sum(), "fully duplicate rows to remove")

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True, drop=True)

In [ ]:
# Keep Label and Attack for splitting/evaluation; never feed either to the model.
df = df.loc[:, ~df.columns.str.lower().str.startswith(('unnamed:', 'index'))]
feature_cols = [c for c in df if c not in ('Label', 'Attack')]

# Protocol identifiers, DNS query types, and FTP return codes are nominal categories.
categorical_cols = [c for c in col_categorical if c in feature_cols]
for col in categorical_cols:
    df[col] = df[col].astype('string').fillna('missing').astype("string[pyarrow]").str.strip()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print(f'{len(numeric_cols)} numeric and {len(categorical_cols)} categorical features')

In [ ]:
print('Categorical:', categorical_cols)
print('Numeric:', numeric_cols)

In [ ]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)    
print(df.isna().any(axis=1).sum(), "rows with at least one NaN to remove")
# df.dropna(inplace=True)

## 5. Optionally reserve an unseen attack

In [ ]:
# Remove incomplete targets and exact repeated rows before any split.
if HOLD_OUT_ATTACK:
    zero_day = df[df['Attack'].eq(HELD_OUT_ATTACK)]
    known = df[df['Attack'].ne(HELD_OUT_ATTACK)]
    print(f'Eligible: {len(known):,} | held-out {HELD_OUT_ATTACK}: {len(zero_day):,}')
else:
    known = df
    zero_day = df.iloc[:0]  # Keep the same schema; no zero-day file is saved.
    print(f'Holdout disabled: all {len(known):,} rows enter the normal split.')


## 6. Split eligible traffic (70% / 15% / 15%)

In [ ]:
# Stratify eligible traffic by Attack; exclude the held-out attack only when enabled.
train, remainder = train_test_split(
    known, test_size=0.30, stratify=known['Attack'], random_state=SEED
)
val, test = train_test_split(
    remainder, test_size=0.50, stratify=remainder['Attack'], random_state=SEED
)
# Drop intermediate frames; retain only the four clean, untransformed splits.
del df, known, remainder
print('train / val / test / zero-day:', *(len(x) for x in (train, val, test, zero_day)))

## Outliers

In [ ]:
# Only numerical measurements—not categories or flag indicators.
outlier_cols = [
    c for c in numeric_cols
    if c not in col_to_one_hot
    and not c.startswith((
        'TCP_FLAGS_',
        'CLIENT_TCP_FLAGS_',
        'SERVER_TCP_FLAGS_',
    ))
]

rows = []

for col in outlier_cols:
    s = train[col]
    finite = s[np.isfinite(s)]

    q1, median, q3, p99, p999 = finite.quantile(
        [0.25, 0.50, 0.75, 0.99, 0.999]
    )
    iqr = q3 - q1

    # Zero IQR makes this rule uninformative for sparse/zero-heavy features.
    iqr_count = (
        ((finite < q1 - 1.5 * iqr) |
         (finite > q3 + 1.5 * iqr)).sum()
        if iqr > 0 else np.nan
    )

    rows.append({
        'feature': col,
        'missing': s.isna().sum(),
        'infinite': np.isinf(s).sum(),
        'minimum': finite.min(),
        'median': median,
        'p99': p99,
        'p99.9': p999,
        'maximum': finite.max(),
        'IQR': iqr,
        'IQR_flagged': iqr_count,
        'above_p99.9': (finite > p999).sum(),
        'outside_float32': (
            (finite > np.finfo(np.float32).max) |
            (finite < -np.finfo(np.float32).max)
        ).sum(),
        'outside_float64': (
            (finite > np.finfo(np.float64).max) |
            (finite < -np.finfo(np.float64).max)
        ).sum(),
    })

outlier_report = pd.DataFrame(rows).set_index('feature')
with pd.option_context('display.max_rows', None):
    display(outlier_report)

## 8. Fit preprocessing on training data only

This CIC notebook previously had no PCA. `PCA_COMPONENTS = None` preserves its existing numeric outputs. To enable PCA over **all scaled numeric measurement columns**, choose an integer component count (no larger than the number of numeric features and fitting rows). Protocol one-hot columns stay outside PCA.

Set `PCA_FIT_ROWS = IMPUTER_FIT_ROWS` for sampled PCA, `None` for full-training PCA, or another positive cap. Pass these into `make_preprocessor(..., pca_fit_rows=PCA_FIT_ROWS, n_components=PCA_COMPONENTS)`. PCA changes the numeric output columns and requires retraining downstream models.

The CIC imputer and numeric scaler retain their existing sampled fitting behavior, now controlled by `IMPUTER_FIT_ROWS`. One-hot vocabularies include all observed training categories. Learned branches are frozen and assembled with one preview row, avoiding a combined full-training feature matrix. All fitting uses training rows only; transforms/writes remain batched. Full-train PCA mode can still require substantial RAM. Requires scikit-learn >= 1.6.

IterativeImputer uses `skip_complete=True` to skip iterative regressions for complete fitting columns. Other fitting settings and optional numeric PCA remain unchanged.


In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def category_fit_frame(data, columns):
    """Fit independent category vocabularies without expanding millions of rows."""
    values = {c: pd.Series(data[c]).drop_duplicates().to_numpy() for c in columns}
    longest = max(map(len, values.values()))
    # Repeat real values to align column lengths; do not invent missing categories.
    return pd.DataFrame({c: np.resize(v, longest) for c, v in values.items()})


def fit_pca(data, columns, preprocessing, n_components=1, fit_rows=None,
            random_state=42, verbose=True):
    """fit_rows=None uses all training rows; an integer caps PCA fitting rows."""
    if fit_rows is not None and fit_rows < 1:
        raise ValueError('fit_rows must be None or a positive row count.')
    if fit_rows is None or fit_rows >= len(data):
        fit_data = data[columns]
    else:
        fit_data = data.sample(n=fit_rows, random_state=random_state)[columns]
    if verbose:
        print(f'PCA fit: {len(fit_data):,}/{len(data):,} training rows; '
              f'{len(columns)} inputs -> {n_components} component(s).', flush=True)
    model = make_pipeline(
        *[step for _, step in preprocessing.steps],
        PCA(n_components=n_components, svd_solver='full'),
        verbose=verbose,
    )
    model.fit(fit_data)
    model.fit_rows_ = len(fit_data)
    if verbose:
        retained = model.steps[-1][1].explained_variance_ratio_.sum()
        print(f'PCA fitted; retained variance: {retained:.2%}.', flush=True)
    return model


def make_preprocessor(train, numeric_cols, one_hot, scaler=None,
                      imputer_fit_rows=500_000, pca_fit_rows=None,
                      n_components=None, random_state=42, verbose=True):
    """Optional PCA over scaled numeric measurements; categories remain separate."""
    sample = train.sample(
        n=min(imputer_fit_rows, len(train)), random_state=random_state,
    )[numeric_cols]
    imputer = IterativeImputer(max_iter=5, random_state=random_state,
                               skip_complete=True, verbose=int(verbose))
    numeric_scaler = StandardScaler() if scaler is None else clone(scaler)
    numbers = make_pipeline(imputer, numeric_scaler, verbose=verbose)
    if verbose:
        print(f'Fitting CIC imputer/scaler on {len(sample):,} training rows...', flush=True)
    numbers.fit(sample)
    imputer_rows = len(sample)
    del sample

    # Preserve the original CIC numeric fit cap. PCA has its own independent cap.
    if n_components is not None:
        numbers_branch = fit_pca(
            train, numeric_cols, preprocessing=make_pipeline(FrozenEstimator(numbers)),
            n_components=n_components, fit_rows=pca_fit_rows,
            random_state=random_state, verbose=verbose,
        )
    else:
        numbers_branch = numbers
        if verbose:
            print('CIC PCA disabled (n_components=None).', flush=True)
    categories = OneHotEncoder(handle_unknown='ignore').fit(
        category_fit_frame(train, one_hot)
    )
    result = ColumnTransformer([
        ('numbers', FrozenEstimator(numbers_branch), numeric_cols),
        ('categories', FrozenEstimator(categories), one_hot),
    ], verbose=verbose)
    result.fit(train.iloc[:1])
    result.imputer_ = imputer
    result.numeric_scaler_ = numeric_scaler
    result.imputer_fit_rows_ = imputer_rows
    result.pca_ = numbers_branch.steps[-1][1] if n_components is not None else None
    if verbose:
        print(f'CIC preprocessor ready: {len(result.get_feature_names_out()):,} features.', flush=True)
    return result


preprocessor = make_preprocessor(
    train, numeric_cols, col_to_one_hot,
    scaler=NUMERIC_SCALER, imputer_fit_rows=IMPUTER_FIT_ROWS,
    pca_fit_rows=PCA_FIT_ROWS, n_components=PCA_COMPONENTS,
    random_state=SEED, verbose=PREPROCESSING_VERBOSE,
)


Save each split below in batches, then release its original frame. To repeat saving or run the optional raw-data benchmarks afterward, rerun the loading/splitting cells first.

## 9. Transform, save, and free each split

In [ ]:
import pyarrow as pa

feature_names = preprocessor.get_feature_names_out().tolist()

original_target_schema = pa.Schema.from_pandas(
    train[['Label', 'Attack']], preserve_index=False
)
print(original_target_schema)
schema = pa.schema(
    [
        (
            name,
            pa.uint8() if name.startswith('categories__')
            else pa.float32()
        )
        for name in feature_names
    ]
    + [
        original_target_schema.field('Label'),
        original_target_schema.field('Attack'),
    ]
)

print(f'Compact output schema:\n{schema}')

import shutil
shutil.rmtree('/kaggle/working/project')

In [ ]:
from datetime import datetime
from time import perf_counter
import gc
import joblib
import sklearn
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.sparse import issparse

PROJECT = Path(OUTPATH)
fitted_imputer = preprocessor.imputer_
fitted_numeric_scaler = preprocessor.numeric_scaler_
fitted_pca = preprocessor.pca_
attack_tag = f"holdout_{HELD_OUT_ATTACK}" if HOLD_OUT_ATTACK else "all_attacks"
RUN_NAME = f"{datetime.now():%Y%m%d_%H%M%S}_{attack_tag}_{type(fitted_imputer).__name__}"
OUTPUT_DIR = PROJECT / 'outputs' / DATASET / 'preprocessing' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 10000  # Reduce this if a transformed batch still uses too much RAM.
print('Saving fitted preprocessor to:', OUTPUT_DIR, flush=True)
joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor.joblib', compress=0)
print('Fitted preprocessor saved.', flush=True)


# Exact baseline from the same cleaned, unencoded TRAIN rows used for export.
import json
raw_category_rows = []
for column in col_to_one_hot:
    unique_values = train[column].drop_duplicates()
    raw_category_rows.append({
        'Raw feature': column,
        'Raw unique entries': len(unique_values),
        'Raw values (first 20)': unique_values.head(20).astype(str).tolist(),
        'Raw missing rows': int(train[column].isna().sum()),
    })
    del unique_values
raw_category_lookup = {r['Raw feature']: r for r in raw_category_rows}
encoded_category_rows = []
for column in preprocessor.get_feature_names_out():
    if not column.startswith('categories__'):
        continue
    inputs = [c for c in col_to_one_hot if column.startswith(f'categories__{c}_')]
    encoded_category_rows.append({
        'Raw feature(s)': ' + '.join(inputs),
        'Raw unique entries': {c: raw_category_lookup[c]['Raw unique entries'] for c in inputs},
        'Raw values (preview)': {c: raw_category_lookup[c]['Raw values (first 20)'] for c in inputs},
        'Encoding': 'One-hot',
        'Encoded column': column,
    })
(OUTPUT_DIR / 'raw_categorical_summary.json').write_text(json.dumps({
    'training_rows': len(train),
    'raw_features': raw_category_rows,
    'encoded_features': encoded_category_rows,
}, indent=2), encoding='utf-8')
del raw_category_rows, raw_category_lookup, encoded_category_rows

# Move frames into the queue; remove the other references so they can be freed.
splits = {'train': train, 'val': val, 'test': test}
if HOLD_OUT_ATTACK:
    splits['zero_day'] = zero_day
del train, val, test, zero_day
counts = []
for name in list(splits):
    rows = splits.pop(name)
    split_started = perf_counter()
    total_batches = (len(rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'Starting {name}: {len(rows):,} rows in {total_batches:,} batches.', flush=True)
    with pq.ParquetWriter(OUTPUT_DIR / f'{name}.parquet', schema, compression='zstd') as writer:
        for start in range(0, len(rows), BATCH_SIZE):
            batch = rows.iloc[start:start + BATCH_SIZE]
            X = preprocessor.transform(batch[feature_cols])
            # # Reduce batch memory before densifying sparse output.
            # # float32 rounding is intentional for compact model inputs.
            # X = X.astype(np.float32, copy=False)
            # Densify this small batch only; never the complete split.
            if issparse(X):
                X = X.toarray()
            prepared = pd.DataFrame(X, columns=feature_names)
            prepared['Label'] = batch['Label'].to_numpy()
            prepared['Attack'] = batch['Attack'].to_numpy()
            table = pa.Table.from_pandas(prepared, schema=schema, preserve_index=False)
            writer.write_table(table)
            batch_number = start // BATCH_SIZE + 1
            if (batch_number == 1 or batch_number == total_batches
                    or batch_number % max(1, PRINT_EVERY_BATCHES) == 0):
                completed = min(start + BATCH_SIZE, len(rows))
                print(f'{name}: batch {batch_number:,}/{total_batches:,}; '
                      f'{completed:,}/{len(rows):,} rows written; '
                      f'{perf_counter() - split_started:.1f}s elapsed.', flush=True)
            del batch, X, prepared, table
    counts.append(f'- {name}: {len(rows):,} rows')
    print(f'Saved {name}: {len(rows):,} rows in {perf_counter() - split_started:.1f}s; releasing raw split.', flush=True)
    del rows
    gc.collect()  # Frees Python references; the allocator may retain some reserved RAM.

holdout_note = f"Reserve `{HELD_OUT_ATTACK}` as zero-day data" if HOLD_OUT_ATTACK else "Use all attack classes; no separate zero-day set"
saved_files = '`train.parquet`, `val.parquet`, `test.parquet`' + (', `zero_day.parquet`' if HOLD_OUT_ATTACK else '')
readme = f"""# {DATASET} preprocessing — {RUN_NAME}

Source: `{DATA_DIR / FILES[DATASET]}`
Holdout enabled: `{HOLD_OUT_ATTACK}`. Held-out attack: `{HELD_OUT_ATTACK if HOLD_OUT_ATTACK else "None (all attacks included)"}`. Seed: `{SEED}`. scikit-learn: `{sklearn.__version__}`.

## Sampling and comparison
Before cleaning and splitting, retain up to {MAX_PER_CLASS:,} random rows per Attack class and all rows from smaller classes. Compare the actual sample to the full original using class proportions, within-class numerical KS distance, and categorical total variation. Diagnostics do not use another reference sample. Validation/test reflect the capped class mix.

## Steps followed
1. Inspect data and attack counts; strip column/attack-name whitespace.
2. Remove dataset identifiers, IP addresses, and generated index columns.
3. Encode the configured CIC categories `{col_to_one_hot}` using training-only one-hot vocabularies; replace numeric infinities with NaN.
4. Remove rows missing targets and exact duplicate rows.
5. {holdout_note}; split eligible traffic 70/15/15, stratified by Attack. With holdout disabled, attack classes are eligible for every split; rare classes are not guaranteed to occur in each split.
6. Fit `{fitted_imputer!r}` and `{fitted_numeric_scaler!r}` on {preprocessor.imputer_fit_rows_:,} sampled training rows (cap {IMPUTER_FIT_ROWS:,}). One-hot vocabularies include all observed training categories. Optional numeric PCA: {PCA_COMPONENTS!r} components; requested PCA row cap: {PCA_FIT_ROWS!r}; actual PCA fit rows: {fitted_pca.n_samples_ if fitted_pca is not None else 'disabled'}. PCA variance retained: {fitted_pca.explained_variance_ratio_.sum() if fitted_pca is not None else 'disabled'}. All fitting uses training data only; branches are frozen and assembled with one preview row.
7. Transform each split in batches of at most {BATCH_SIZE:,} rows and write directly to Parquet with Zstandard compression.
8. Release each batch and completed split before continuing.

## Saved files
`raw_categorical_summary.json` records raw training categorical domains and encoded-column mappings. The imbalance cells save `adasyn_generated_preview.parquet` with genuine generated rows and Tomek retention flags; its metadata fields are not model inputs.
{saved_files} contain TRANSFORMED numeric feature columns plus `Label` and `Attack`.
Features are already imputed, encoded, and scaled: do not transform them again.
`preprocessor.joblib` is for new RAW data only. Feature order is `preprocessor.get_feature_names_out()`.
Storage types: scaled numerical features or optional numeric PCA outputs use float32; one-hot features use uint8. Protocol one-hot outputs remain outside PCA. IterativeImputer skips iterative regressions for complete fitting columns.
Label and Attack retain their original Arrow types. The same schema is used for every batch and split.
Float32 reduces precision compared with float64.
These files differ from older exports of cleaned, untransformed features.

{chr(10).join(counts)}

## Resampling and diagnostics
After preprocessing/export, ADASYN operates on prepared training data, then Tomek Links cleans the original majority Attack class. Validation/test are not resampled. Synthetic feature coordinates remain floating point. The resampling cells report start/end times, shapes, matrix size and row deltas; no periodic status messages are printed. Tomek Links, result tables and Plotly are separate cells. Enable RUN_CATEGORY_DIAGNOSTICS in Testing to compare exact raw/encoded/resampled categorical domains and inspect the saved genuine generated-row preview.

## Read prepared batches
```python
import pyarrow.parquet as pq
for record_batch in pq.ParquetFile('train.parquet').iter_batches(batch_size=10000):
    batch = record_batch.to_pandas()
    X = batch.drop(columns=['Label', 'Attack'])
    y = batch['Label']
    # Consume this batch directly; do not apply preprocessor.transform(X).
```

Initial loading/splitting and imputer fitting still require RAM. Full-train PCA can require substantial RAM; set PCA_FIT_ROWS to a smaller cap. To reload preprocessor.joblib, use scikit-learn >= 1.6; no custom transformer class is required. Saving releases the original split frames; rerun loading/splitting before using raw-data benchmarks or repeating the save step.
Row splits and exact deduplication do not guarantee session isolation or isolation of identical predictors with conflicting labels.
"""
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print('Prepared datasets saved to:', OUTPUT_DIR)

## 7. Class Imbalance

Load the processed `train.parquet`, resample its `Attack` classes with ADASYN, then clean the original majority Attack class with Tomek Links. Exclude both targets from features. Restore binary `Label` from the original Attack-to-Label mapping when saving. Validation and test are not resampled here. Synthetic one-hot coordinates can be fractional and remain floating point.

In [ ]:
HANDEL_IMBALANCE = True
ADASYN_BALANCE_TOLERANCE = 0.03  # Leave classes within 3% unchanged.

from contextlib import contextmanager
from time import perf_counter

@contextmanager
def resampling_status(stage):
    """Report stage start and total elapsed time without periodic messages."""
    started = perf_counter()
    print(f'{stage}: starting...', flush=True)
    finished = False
    try:
        yield
        finished = True
    finally:
        status = 'finished' if finished else 'stopped before completion'
        print(f'{stage}: {status} | {perf_counter() - started:.1f}s elapsed',
              flush=True)


In [ ]:
if HANDEL_IMBALANCE:
    %pip install -q imbalanced-learn plotly

    import gc
    import numpy as np
    import pandas as pd
    import plotly.express as px
    from imblearn.over_sampling import ADASYN
    from imblearn.under_sampling import TomekLinks

    prepared_train = pd.read_parquet(OUTPUT_DIR / 'train.parquet')
    original_columns = prepared_train.columns.tolist()
    target_dtypes = prepared_train[['Label', 'Attack']].dtypes

    # Every Attack class must correspond to exactly one binary Label.
    target_pairs = prepared_train[['Attack', 'Label']].drop_duplicates()
    if target_pairs['Attack'].duplicated().any():
        raise ValueError('An Attack class has multiple Label values. Fix this mapping first.')
    attack_to_label = target_pairs.set_index('Attack')['Label']

    resampling_features = [
        c for c in prepared_train.columns if c not in ('Label', 'Attack')
    ]
    # Floating point preserves synthetic fractional one-hot values.
    X_train = prepared_train[resampling_features].to_numpy(dtype=np.float32)
    y_train = prepared_train['Attack'].to_numpy(copy=True)

    del prepared_train, target_pairs
    gc.collect()

    before_counts = pd.Series(y_train).value_counts().sort_index()
    majority_attack = before_counts.idxmax()

    print('Before resampling (Attack):')
    display(before_counts.rename_axis('Attack').rename('rows').to_frame())

ADASYN uses an explicit dictionary of desired class counts. Classes already within `ADASYN_BALANCE_TOLERANCE = 0.03` (3%) of the largest class are left unchanged. This avoids requesting a negligible top-up such as 29 samples across 209,971 Reconnaissance rows, where every per-row allocation can round to zero.

The 3% tolerance is an explicit policy, not a requirement of ADASYN. Smaller classes target the largest count; the generated counts remain approximate. There is no silent fallback to another sampler. Independent ADASYN failures, such as unsuitable neighborhoods, remain visible.

### Runtime reporting

ADASYN and Tomek Links do not expose built-in row-progress callbacks, percentages or ETA. Report stage start, finish/interruption and total elapsed seconds only; no periodic messages are printed. Feature/target shapes, post-ADASYN matrix size, added/removed row counts and the timing summary remain available. ADASYN's skipped case is marked explicitly with zero sampler time.

The samplers still run on the full training matrix. No independent resampling batches, changed neighbour search or approximate cleaning are introduced. Tomek Links, the result tables and the Plotly chart are separate cells; rerun the tables/chart without repeating resampling while their small result variables remain in the session.


### ADASYN — timing and output shape


In [ ]:
if HANDEL_IMBALANCE:
    input_train_rows = len(y_train)
    print(f'Before ADASYN: X={X_train.shape}, y={y_train.shape}', flush=True)
    adasyn_seconds = 0.0
    adasyn_skipped = False
    target_count = int(before_counts.max())
    sampling_targets = {
        attack: target_count
        for attack, count in before_counts.items()
        if count < (1 - ADASYN_BALANCE_TOLERANCE) * target_count
    }

    if sampling_targets:
        smallest_target_class = min(
            int(before_counts[attack]) for attack in sampling_targets
        )
        if smallest_target_class < 2:
            raise ValueError(
                'ADASYN needs at least two examples in each targeted Attack class.'
            )

        print('ADASYN desired class counts:', sampling_targets, flush=True)
        adasyn = ADASYN(
            sampling_strategy=sampling_targets,
            n_neighbors=min(5, smallest_target_class - 1),
            random_state=SEED,
        )
        adasyn_started = perf_counter()
        with resampling_status('ADASYN'):
            X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)
        adasyn_seconds = perf_counter() - adasyn_started
    else:
        adasyn_skipped = True
        print('Classes are already approximately balanced; skipping ADASYN.', flush=True)
        X_adasyn, y_adasyn = X_train, y_train

    adasyn_rows = len(y_adasyn)
    print(f'After ADASYN: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
    print(f'ADASYN time: {adasyn_seconds:.1f}s | '
          f'Rows added: {adasyn_rows - input_train_rows:,}', flush=True)
    print(f'Post-ADASYN feature matrix: {X_adasyn.nbytes / 2**30:.2f} GiB | '
          f'dtype={X_adasyn.dtype}', flush=True)

    adasyn_counts = pd.Series(y_adasyn).value_counts().sort_index()
    print('After ADASYN (Attack):')
    display(pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
    }).fillna(0).astype(int).rename_axis('Attack'))

    # ADASYN appends generated rows after the original training rows.
    # Keep a small, genuine generated preview before Tomek/cleanup releases them.
    preview_indices = []
    for attack in sampling_targets:
        generated_positions = np.flatnonzero(y_adasyn[input_train_rows:] == attack)
        preview_indices.extend((input_train_rows + generated_positions[:5]).tolist())
    preview_features = [c for c in resampling_features if c.startswith('categories__')]
    preview_feature_indices = [resampling_features.index(c) for c in preview_features]
    generated_preview = pd.DataFrame(
        X_adasyn[np.ix_(preview_indices, preview_feature_indices)], columns=preview_features,
    )
    generated_preview.insert(0, 'ADASYN row', preview_indices)
    generated_preview['Attack'] = y_adasyn[preview_indices]
    generated_preview['Label'] = generated_preview['Attack'].map(attack_to_label)
    # Tomek updates this column after its selection is known.
    generated_preview['Kept after Tomek'] = pd.Series(
        [None] * len(generated_preview), dtype='boolean',
    )
    generated_preview.to_parquet(
        OUTPUT_DIR / 'adasyn_generated_preview.parquet', index=False,
    )
    del preview_indices, preview_features, preview_feature_indices
    if sampling_targets:
        del generated_positions

    del X_train, y_train
    gc.collect()

### Tomek Links — timing and output shape


In [ ]:
if HANDEL_IMBALANCE:
    print(f'Before Tomek Links: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
    # Preserve the original policy: clean only the original majority class.
    tomek = TomekLinks(
        sampling_strategy=[majority_attack],
        n_jobs=-1,
    )
    tomek_started = perf_counter()
    with resampling_status('Tomek Links'):
        X_train_balanced, y_train_balanced = tomek.fit_resample(
            X_adasyn, y_adasyn
        )
    tomek_seconds = perf_counter() - tomek_started
    tomek_removed_rows = len(y_adasyn) - len(y_train_balanced)
    print(f'After Tomek Links: X={X_train_balanced.shape}, '
          f'y={y_train_balanced.shape}', flush=True)
    print(f'Tomek Links time: {tomek_seconds:.1f}s | '
          f'Rows removed: {tomek_removed_rows:,}', flush=True)
    after_counts = pd.Series(y_train_balanced).value_counts()

    # Record which preview rows survived, using Tomek's actual selected indices.
    generated_preview['Kept after Tomek'] = np.isin(
        generated_preview['ADASYN row'].to_numpy(), tomek.sample_indices_,
    )
    generated_preview.to_parquet(
        OUTPUT_DIR / 'adasyn_generated_preview.parquet', index=False,
    )
    del generated_preview

    # Free the expanded input before plotting or exporting the balanced output.
    del X_adasyn, y_adasyn
    gc.collect()


### Resampling results and timings


In [ ]:
if HANDEL_IMBALANCE:
    distribution = pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
        'After Tomek Links': after_counts,
    }).fillna(0).astype(int).rename_axis('Attack')

    distribution['Before (%)'] = (
        100 * distribution['Before'] / distribution['Before'].sum()
    ).round(2)
    distribution['After (%)'] = (
        100 * distribution['After Tomek Links']
        / distribution['After Tomek Links'].sum()
    ).round(2)
    display(distribution)

    resampling_summary = pd.DataFrame([
        {'Stage': 'ADASYN', 'Status': 'skipped' if adasyn_skipped else 'finished',
         'Input rows': input_train_rows, 'Output rows': adasyn_rows,
         'Features': len(resampling_features), 'Seconds': adasyn_seconds},
        {'Stage': 'Tomek Links', 'Status': 'finished',
         'Input rows': adasyn_rows, 'Output rows': adasyn_rows - tomek_removed_rows,
         'Features': len(resampling_features), 'Seconds': tomek_seconds},
    ]).set_index('Stage')
    display(resampling_summary.round({'Seconds': 2}))


### Plot class distribution

Run this cell again to redraw the chart without rerunning Tomek Links.


In [ ]:
if HANDEL_IMBALANCE:
    # Plot aggregated counts instead of passing millions of rows to Plotly.
    plot_counts = distribution[
        ['Before', 'After ADASYN', 'After Tomek Links']
    ].reset_index().melt(
        id_vars='Attack', var_name='Stage', value_name='Rows'
    )
    plot_counts['Attack'] = plot_counts['Attack'].astype(str)
    fig = px.histogram(
        plot_counts,
        x='Attack',
        y='Rows',
        color='Stage',
        histfunc='sum',
        barmode='group',
        title='Attack class distribution before and after resampling',
        labels={'Rows': 'Number of samples'},
        category_orders={'Attack': distribution.index.astype(str).tolist()},
    )
    fig.update_layout(yaxis_title='Number of samples', xaxis_tickangle=-45)
    fig.show()



In [ ]:
if HANDEL_IMBALANCE:
    balanced_train = pd.DataFrame(
        X_train_balanced,
        columns=resampling_features,
    )
    balanced_train['Attack'] = pd.Series(y_train_balanced).astype(
        target_dtypes['Attack']
    )
    balanced_train['Label'] = balanced_train['Attack'].map(
        attack_to_label
    ).astype(target_dtypes['Label'])
    balanced_train = balanced_train[original_columns]

    balanced_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
    balanced_train.to_parquet(
        balanced_path,
        index=False,
        compression='zstd',
    )
    print('Saved:', balanced_path)

    del balanced_train, X_train_balanced, y_train_balanced
    gc.collect()

Train using `train_adasyn_tomek.parquet`, with the existing `val.parquet` and `test.parquet`. The saved training file contains both `Attack` and `Label`; use `Attack` as the multiclass target. Exclude both targets from the model inputs. These features are already preprocessed—do not transform them again.

The table and Plotly histograms above show counts before resampling, after ADASYN, and after Tomek Links. Percentages compare the original and final training distributions.

## Github

# Testing

The categorical diagnostic below reads the main saved output run. Run the platform/path/dataset configuration and set `OUTPUT_DIR` to that run, or continue after the main export/resampling cells. No raw-data benchmark or refitting is required. The independent flag controls the test; missing diagnostic snapshots produce prerequisite guidance.


In [ ]:
# Enable either or both groups, then run the loading cell immediately below.
RUN_TESTING = True
RUN_CATEGORY_DIAGNOSTICS = True  # Main raw -> encoded -> resampled categorical audit.


### Categorical unique values and generated samples

Enable `RUN_CATEGORY_DIAGNOSTICS` and use `OUTPUT_DIR` from the main run. This cell reads main saved outputs; it does not refit preprocessing or rerun ADASYN/Tomek.

- Before processing: exact unique counts (including missing) and up to 20 examples from the same cleaned, unencoded training `Protocol` column.
- After processing: exact unique counts and value previews for each `Protocol` one-hot output in `train.parquet`.
- After resampling: exact unique counts, new values and non-binary indicator entries in `train_adasyn_tomek.parquet`, after ADASYN and Tomek Links. `MAX_UNIQUE_EXAMPLES = 12` limits displayed encoded examples; counts use every row. One Parquet column is read at a time to limit memory; full scans can take time.
- Generated samples: up to five genuine appended ADASYN rows per targeted Attack, with encoded coordinates and whether Tomek retained each row. A small preview may miss fractional values found by the full final-domain comparison.

One-hot indicators normally contain 0/1; ADASYN can interpolate fractional coordinates. The diagnostic displays values without rounding or modifying data. Optional CIC PCA transforms only numeric measurements; `Protocol` one-hot outputs stay separate and remain the focus of this test. This compares value domains, not category frequencies or semantic validity.

Run the updated main export and resampling cells to create `raw_categorical_summary.json` and `adasyn_generated_preview.parquet`. Older runs do not contain these snapshots. If raw frames/matrices were released, recreate them through the usual loading/splitting/fitting/export flow before resampling. Arbitrary final rows are never labelled as generated.


In [ ]:
if RUN_CATEGORY_DIAGNOSTICS:
    import json
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq

    # OUTPUT_DIR must point to the same run for all three diagnostic files.
    raw_summary_path = OUTPUT_DIR / 'raw_categorical_summary.json'
    processed_path = OUTPUT_DIR / 'train.parquet'
    resampled_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
    preview_path = OUTPUT_DIR / 'adasyn_generated_preview.parquet'
    MAX_UNIQUE_EXAMPLES = 12

    if not all(p.exists() for p in [raw_summary_path, processed_path,
                                    resampled_path, preview_path]):
        print('Run the updated main export, ADASYN and Tomek Links cells first. '
              'They create the categorical baseline and generated-row preview '
              'inside OUTPUT_DIR. This test does not rerun resampling.')
    else:
        category_baseline = json.loads(raw_summary_path.read_text())
        raw_category_uniques = pd.DataFrame(category_baseline['raw_features'])
        print('Raw categorical values from the SAME training split before processing:')
        display(raw_category_uniques)
        print('Rows compared: raw/processed training = '
              f"{category_baseline['training_rows']:,}; final resampled training = "
              f'{pq.ParquetFile(resampled_path).metadata.num_rows:,}')

        # Read one encoded column at a time, not either complete feature matrix.
        comparison_rows = []
        for mapping in category_baseline['encoded_features']:
            column = mapping['Encoded column']
            before_unique = pc.unique(pq.read_table(
                processed_path, columns=[column]
            ).column(0).cast(pa.float64()))
            after_unique = pc.unique(pq.read_table(
                resampled_path, columns=[column]
            ).column(0).cast(pa.float64()))
            new_unique = pc.filter(
                after_unique,
                pc.invert(pc.is_in(after_unique, value_set=before_unique)),
            )
            non_binary = int(pc.sum(pc.and_(
                pc.not_equal(after_unique, 0),
                pc.not_equal(after_unique, 1),
            )).as_py() or 0)
            comparison_rows.append({
                **mapping,
                'Unique after processing': len(before_unique),
                'Processed values (preview)': before_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'Unique after ADASYN + Tomek': len(after_unique),
                'Resampled values (preview)': after_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'New unique entries': len(new_unique),
                'New values (preview)': new_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'Non-binary unique entries (final)': non_binary,
            })
            del before_unique, after_unique, new_unique

        category_unique_comparison = pd.DataFrame(comparison_rows)
        print('Before processing -> encoded training -> final resampled training:')
        display(category_unique_comparison)
        print('Indicator columns containing values other than 0 and 1 after resampling:')
        display(category_unique_comparison.loc[
            category_unique_comparison['Non-binary unique entries (final)'].fillna(0).gt(0)
        ].head(30))

        generated_samples = pd.read_parquet(preview_path)
        if generated_samples.empty:
            print('ADASYN generated no rows in this run; the generated preview is empty.')
        else:
            encoded_columns = [m['Encoded column'] for m in category_baseline['encoded_features']]
            indicator_columns = encoded_columns
            generated_encoded_values = generated_samples.melt(
                id_vars=['ADASYN row', 'Attack', 'Label', 'Kept after Tomek'],
                value_vars=indicator_columns,
                var_name='Encoded column', value_name='Generated value',
            )
            generated_fractional_examples = generated_encoded_values.loc[
                ~generated_encoded_values['Generated value'].isin([0, 1])
            ]
            focus_columns = list(dict.fromkeys(
                generated_fractional_examples['Encoded column'].tolist()
                + encoded_columns
            ))[:12]
            print('Genuine ADASYN-generated rows (up to 5 per oversampled Attack):')
            display(generated_samples[
                ['ADASYN row', 'Attack', 'Label', 'Kept after Tomek'] + focus_columns
            ].head(20))
            print('Generated non-binary indicator coordinates (first 40):')
            display(generated_fractional_examples.head(40))
            if generated_fractional_examples.empty:
                print('No fractional indicator values occurred in this small preview. '
                      'Use the full-data unique comparison above for all final rows.')
